# Predição: Prova de Fotos 

Autor:  Viviane da Rosa Sommer

Atualizado: 26/12/2024

## Objetivo:
Notebook para fazer a predição do modelo de Coral-Sol (MSO-V1) nas fotos dos vídeos das provas de fogo.

Foi feita uma amostragem de 30 imagens positivas e 30 imagens negativas para cada uma das provas de foto.

## Importações Necessárias

In [ ]:
!pip install ultralytics
!pip install opencv-python-headless
!pip install torch
!pip install tqdm
!pip install pandas

import glob
import random
import cv2
import numpy as np
import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
import torchvision
import torch

## Declaração de Constantes e Modelos

In [ ]:
CORAL_CLASS_ID = 0
RESIZED_DIM_CORAL = 800
SAMPLE_SIZE = 30 

coral_model = YOLO('../Coral_Note/runs/segment/Lote123-SOverlay/train-800-lote123/weights/best.pt')

## Funções Necessárias

In [ ]:
def read_image(filename: str) -> np.ndarray:
    """
    Read an image from a file and convert it to a NumPy array.

    Args:
        filename (str): The path to the image file.

    Returns:
        np.ndarray: The image as a NumPy array.
    """
    image = cv2.imread(filename)
    if image is None:
        raise ValueError(f"Não foi possível carregar a imagem: {filename}")
    height, width, _ = image.shape
    mid = height // 2
    top = max(0, mid - int(0.34 * height))
    bottom = min(height, mid + int(0.34 * height))
    cropped_image = image[top:bottom, :]

    return cropped_image


def process_results(results: list) -> any:
    """
    Processes the results obtained from the model, returning the first valid result with masks.

    Parameters:
        results (list): List of model detection results.

    Returns:
        any: The first valid result containing masks, or None if no valid result is found.
    """
    for result in results:
        if result.masks is not None:
            return result
    return None

                    
def prediction_coral(img: np.array) -> tuple:
    """
    Predicts coral objects using the coral model and returns a binary mask for coral regions.

    Parameters:
        img (np.array): Input image for prediction.

    Returns:
        tuple: A tuple containing:
            - centered_mask (np.ndarray): Resized coral mask with centered annotations.
            - torch.Tensor: The centered mask as a PyTorch tensor.
    """
    original_height, original_width = img.shape[:2]
    coral_results = coral_model(img, verbose=False)
    coral_best_result = process_results(coral_results)
    
    if coral_best_result is not None and coral_best_result.masks is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]

        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.5))[0]
        coral_boxes = boxes[coral_indices]
        coral_masks = masks[coral_indices]
        coral_scores = scores[coral_indices]

        if len(coral_boxes) > 0:
            nms_indices = torchvision.ops.nms(coral_boxes[:, :4], coral_scores, iou_threshold=0.2)
            coral_boxes = coral_boxes[nms_indices]
            coral_masks = coral_masks[nms_indices]
            final_coral_mask = torch.any(coral_masks, dim=0).int() * 255
        else:
            final_coral_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
    else:
        final_coral_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
    
    final_coral_mask_np = final_coral_mask.cpu().numpy()
    mask_height, mask_width = final_coral_mask_np.shape

    resized_final_coral_mask = cv2.resize(final_coral_mask_np, (original_width, original_height), interpolation=cv2.INTER_NEAREST)
    centered_mask = np.zeros((original_height, original_width), dtype=np.uint8)
    y_offset = (original_height - resized_final_coral_mask.shape[0]) // 2
    x_offset = (original_width - resized_final_coral_mask.shape[1]) // 2
    if (y_offset >= 0 and x_offset >= 0 and 
        y_offset + resized_final_coral_mask.shape[0] <= original_height and 
        x_offset + resized_final_coral_mask.shape[1] <= original_width):
        
        centered_mask[y_offset:y_offset + resized_final_coral_mask.shape[0], 
                      x_offset:x_offset + resized_final_coral_mask.shape[1]] = resized_final_coral_mask
    else:
        centered_mask = resized_final_coral_mask.copy()

    return centered_mask, torch.from_numpy(centered_mask).int()


def plot_images(img: np.ndarray, 
                intersection_mask: np.ndarray) -> None:
    """
    Displays the original image and the annotated image side by side.

    Parameters:
        img (np.ndarray): The original image in BGR format (NumPy array).
        intersection_mask (np.ndarray): A mask indicating intersections between model and expert annotations.
        title (str): Title for the annotated image display (default: "Annotation").

    Description:
        This function performs the following operations:
        1. Displays the original image side by side with the annotated image.
        2. Annotates the image by drawing contours from the provided mask (`intersection_mask`).

    Returns:
        None: The function does not return any value; it only displays the images.
    """
    annotated_img = img.copy()
    
    if intersection_mask.any():
        contours, _ = cv2.findContours(intersection_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(annotated_img, contours, -1, (0, 0, 255), thickness=4)  
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
    title = "Positive" if os.path.basename(os.path.dirname(filename)) != "NEGATIVA" else "Negative"

    
    plt.figure(figsize=(15, 7))
    
    plt.subplot(1, 2, 1)
    plt.imshow(img_rgb)
    plt.title(f"Expert Annotated Image - {title}")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(annotated_img_rgb)
    plt.title(f"Model Annotated Image")
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
        

## Prova de Fotos - OS 6000696049

In [ ]:
INPUT_DIRECTORY_POSITIVE_IMAGES = "Provas-de-foto/Prova de foto _6000696049/POSITIVA"
INPUT_DIRECTORY_NEGATIVE_IMAGES = "Provas-de-foto/Prova de foto _6000696049/NEGATIVA"

positive_image_files = [
    filename for filename in glob.glob(f"{INPUT_DIRECTORY_POSITIVE_IMAGES}/**.png")
]

negative_image_files = [
    filename for filename in glob.glob(f"{INPUT_DIRECTORY_NEGATIVE_IMAGES}/**.png")
]

positive_sampled_files = random.sample(positive_image_files, min(SAMPLE_SIZE, len(positive_image_files)))
negative_sampled_files = random.sample(negative_image_files, min(SAMPLE_SIZE, len(negative_image_files)))

for i, filename in enumerate(positive_sampled_files):
    try:
        img = read_image(filename)
        if img is None:
            continue
        coral_mask_resized, coral_mask_tensor = prediction_coral(img)
        plot_images(img, coral_mask_resized)
    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        continue

for i, filename in enumerate(negative_sampled_files):
    try:
        img = read_image(filename)
        if img is None:
            continue
        coral_mask_resized, coral_mask_tensor = prediction_coral(img)
        plot_images(img, coral_mask_resized)
    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        continue

## Prova de Fotos - OS 6000504195

In [ ]:
INPUT_DIRECTORY_POSITIVE_IMAGES = "Provas-de-foto/Prova de foto_6000504195/POSITIVA"
INPUT_DIRECTORY_NEGATIVE_IMAGES = "Provas-de-foto/Prova de foto_6000504195/NEGATIVA"

positive_image_files = [
    filename for filename in glob.glob(f"{INPUT_DIRECTORY_POSITIVE_IMAGES}/**.png")
]

negative_image_files = [
    filename for filename in glob.glob(f"{INPUT_DIRECTORY_NEGATIVE_IMAGES}/**.png")
]

positive_sampled_files = random.sample(positive_image_files, min(SAMPLE_SIZE, len(positive_image_files)))
negative_sampled_files = random.sample(negative_image_files, min(SAMPLE_SIZE, len(negative_image_files)))

for i, filename in enumerate(positive_sampled_files):
    try:
        img = read_image(filename)
        if img is None:
            continue
        coral_mask_resized, coral_mask_tensor = prediction_coral(img)
        plot_images(img, coral_mask_resized)
    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        continue

for i, filename in enumerate(negative_sampled_files):
    try:
        img = read_image(filename)
        if img is None:
            continue
        coral_mask_resized, coral_mask_tensor = prediction_coral(img)
        plot_images(img, coral_mask_resized)
    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        continue

## Prova de Fotos - OS 6000701944

In [ ]:
INPUT_DIRECTORY_POSITIVE_IMAGES = "Provas-de-foto/Prova de Foto_6000701944/POSITIVA"
INPUT_DIRECTORY_NEGATIVE_IMAGES = "Provas-de-foto/Prova de Foto_6000701944/NEGATIVA"

positive_image_files = [
    filename for filename in glob.glob(f"{INPUT_DIRECTORY_POSITIVE_IMAGES}/**.png")
]

negative_image_files = [
    filename for filename in glob.glob(f"{INPUT_DIRECTORY_NEGATIVE_IMAGES}/**.png")
]

positive_sampled_files = random.sample(positive_image_files, min(SAMPLE_SIZE, len(positive_image_files)))
negative_sampled_files = random.sample(negative_image_files, min(SAMPLE_SIZE, len(negative_image_files)))

for i, filename in enumerate(positive_sampled_files):
    try:
        img = read_image(filename)
        if img is None:
            continue
        coral_mask_resized, coral_mask_tensor = prediction_coral(img)
        plot_images(img, coral_mask_resized)
    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        continue

for i, filename in enumerate(negative_sampled_files):
    try:
        img = read_image(filename)
        if img is None:
            continue
        coral_mask_resized, coral_mask_tensor = prediction_coral(img)
        plot_images(img, coral_mask_resized)
    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        continue

In [ ]:
!jupyter nbconvert --to html prova-de-foto-modelo-MSOV1.ipynb